# MNIST/Fashion-MNIST Projection Experiments

This notebook explores random projections on MNIST and Fashion-MNIST datasets.

**Experiments:**
1. PCA vs Random Projection comparison
2. Rectangle vs Square projection matrices
3. Multi-layer RP + ReLU transformations
4. Johnson-Lindenstrauss comparison

In [ ]:
# Setup
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import torch

from rp_study.config import ExperimentConfig
from rp_study.data.loaders import load_fashion_mnist, load_mnist, get_data_loader
from rp_study.projections import (
    random_projection_matrix, 
    multi_layer_projection,
    apply_random_projection,
    jl_projection
)
from rp_study.visualization.projection_plots import (
    plot_pca_vs_rp,
    plot_multi_layer_transformations,
    plot_jl_vs_rp
)

# Setup seeds
config = ExperimentConfig(seed=42)
config.setup_seeds()

DEVICE = config.get_device()
print(f"Using device: {DEVICE}")

## Configuration

Modify these parameters to customize the experiments:

In [ ]:
# Dataset configuration
DATASET = "fashion_mnist"  # Options: "mnist", "fashion_mnist"
DATA_DIR = "../data"

# Projection parameters
TARGET_DIM = 2  # Target dimension for visualization

# Multi-layer experiment parameters
LAYER_COUNTS = [1, 5, 10, 20]

## 1. Load Dataset

In [ ]:
# Load data
X, y = get_data_loader(
    dataset_name=DATASET,
    data_dir=DATA_DIR,
    train=True,
    flatten=True,
    as_numpy=True
)

print(f"Dataset: {DATASET}")
print(f"Data shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Number of classes: {len(np.unique(y))}")

## 2. PCA vs Random Projection

Compare dimensionality reduction methods.

In [ ]:
def relu(data):
    return np.maximum(0, data)

# PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# Random Projection (rectangular: d -> 2)
d = X.shape[1]
R = random_projection_matrix(d, 2, variance="he", as_numpy=True)
X_rp = X @ R

# Apply ReLU
X_pca_relu = relu(X_pca)
X_rp_relu = relu(X_rp)

# Plot comparison
fig = plot_pca_vs_rp(
    X, X_pca, X_rp, 
    labels=y,
    X_pca_relu=X_pca_relu,
    X_rp_relu=X_rp_relu,
    title=f"{DATASET.upper()}: PCA vs Random Projection"
)
plt.show()

## 3. Rectangle vs Square Projections

Compare different projection matrix shapes:
- **Rectangle**: Projects directly from d -> 2 dimensions
- **Square**: Projects d -> d, then visualize with PCA

In [ ]:
# Square projection (d -> d)
R_square = random_projection_matrix(d, d, variance="he", as_numpy=True)
X_square = X @ R_square
X_square_relu = relu(X_square)

# Reduce for visualization
pca_square = PCA(n_components=2)
X_square_vis = pca_square.fit_transform(X_square)
X_square_relu_vis = pca_square.fit_transform(X_square_relu)

# Plot comparison
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
scatter_kwargs = dict(s=5, c=y, cmap='viridis', alpha=0.7)

axes[0, 0].scatter(X_pca[:, 0], X_pca[:, 1], **scatter_kwargs)
axes[0, 0].set_title("PCA (baseline)")
axes[0, 0].axis('equal')

axes[0, 1].scatter(X_rp[:, 0], X_rp[:, 1], **scatter_kwargs)
axes[0, 1].set_title("RP Rectangular (d→2)")
axes[0, 1].axis('equal')

axes[0, 2].scatter(X_square_vis[:, 0], X_square_vis[:, 1], **scatter_kwargs)
axes[0, 2].set_title("RP Square (d→d) + PCA")
axes[0, 2].axis('equal')

axes[1, 0].scatter(X_pca_relu[:, 0], X_pca_relu[:, 1], **scatter_kwargs)
axes[1, 0].set_title("PCA + ReLU")
axes[1, 0].axis('equal')

axes[1, 1].scatter(X_rp_relu[:, 0], X_rp_relu[:, 1], **scatter_kwargs)
axes[1, 1].set_title("RP Rectangular + ReLU")
axes[1, 1].axis('equal')

axes[1, 2].scatter(X_square_relu_vis[:, 0], X_square_relu_vis[:, 1], **scatter_kwargs)
axes[1, 2].set_title("RP Square + ReLU + PCA")
axes[1, 2].axis('equal')

plt.suptitle(f"{DATASET.upper()}: Rectangle vs Square Projections", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Multi-Layer RP + ReLU

Apply multiple layers of random projections with ReLU activation.

In [ ]:
# Multi-layer rectangular (projects to 2D)
print("Running multi-layer rectangular projections...")
results_rect = {}
for n_layers in LAYER_COUNTS:
    X_proj = multi_layer_projection(X, n_layers, mode="rectangular", output_dim=2)
    results_rect[n_layers] = X_proj
    print(f"  {n_layers} layers: shape {X_proj.shape}")

# Plot
fig = plot_multi_layer_transformations(
    X, results_rect, labels=y, mode="rectangular",
    suptitle=f"{DATASET.upper()}: Multi-Layer Rectangular RP + ReLU"
)
plt.show()

In [ ]:
# Multi-layer square (preserves dimension, visualize with PCA)
print("Running multi-layer square projections...")
results_square = {}
for n_layers in LAYER_COUNTS:
    X_proj = multi_layer_projection(X, n_layers, mode="square")
    # Reduce for visualization
    pca_temp = PCA(n_components=2)
    X_vis = pca_temp.fit_transform(X_proj)
    results_square[n_layers] = X_vis
    print(f"  {n_layers} layers: projected shape {X_proj.shape}, visualized shape {X_vis.shape}")

# Plot
fig = plot_multi_layer_transformations(
    X, results_square, labels=y, mode="square",
    suptitle=f"{DATASET.upper()}: Multi-Layer Square RP + ReLU (PCA for viz)"
)
plt.show()

## 4b. Comparing Initialization Strategies on Multi-Layer Square RP + ReLU

Compare how different weight initialization strategies affect the geometry after multiple layers.

### Baseline Strategies:
1. **Uniform**: Weights from U(-a, a) where a = sqrt(3 * 2/d) so variance = 2/d
2. **He (Gaussian)**: Weights from N(0, sqrt(2/d)) - standard He initialization  
3. **Row-Centered**: He + column-centered (full centering, causes gradient trap)
4. **Row-Centered (Var Adj)**: Row-centered + rescale to restore He variance

### Experimental Strategies (to address gradient trap):
5. **Partial Centered (α=0.5)**: Soft centering with `W = W - α * mean(W)` where α < 1
   - Provides partial geometric benefit without full gradient constraint
6. **Orthogonal He**: QR-based orthogonal initialization scaled for ReLU
   - Rows are orthogonal but DON'T sum to zero (no gradient trap)
7. **Centered + DC**: Full centering + add small DC component back
   - Breaks zero-sum constraint while keeping most centering benefit

### Key Insights:
- **Geometric collapse** is caused by ReLU shrinking angles between vectors
- **Row-centering** can prevent collapse but creates a "gradient trap" (∑_j W[i,j] = 0 forces ∑_j δ_j = 0)
- The experimental strategies aim to get geometric benefits without the gradient penalty

In [ ]:
from rp_study.projections import multi_layer_rp_with_init
from rp_study.models.initializers import list_initializers
import torch

# Name mapping: notebook short names -> registry names
INIT_NAME_MAP = {
    "uniform": "uniform_he",
    "he": "he",
    "row_centered": "row_centered_he",
    "row_centered_var_adj": "row_centered_he_var_adj",
    "partial_centered": "partial_centered_he",
    "orthogonal_he": "orthogonal_he",
    "orthogonal_tuned": "orthogonal_tuned",
    "centered_with_dc": "centered_with_dc_he",
    "kernel_preserving": "kernel_preserving",
    "row_centered_final": "row_centered_final",
}


def multi_layer_square_with_init(X, num_layers, init_type="uniform", **kwargs):
    """Apply multi-layer RP + ReLU using registry initializers.
    
    Thin wrapper around multi_layer_rp_with_init() that maps notebook
    short names to registry names.
    """
    registry_name = INIT_NAME_MAP.get(init_type, init_type)
    return multi_layer_rp_with_init(X, num_layers, init_strategy=registry_name, **kwargs)


print(f"Available initializers from registry: {list_initializers()}")
print(f"\nNotebook name -> Registry name mapping:")
for short, full in INIT_NAME_MAP.items():
    print(f"  {short:25s} -> {full}")

In [ ]:
# Compare initialization strategies across different layer counts
INIT_TYPES = [
    "uniform", 
    "he", 
    "row_centered",
    "row_centered_var_adj",
    "partial_centered",      # Option 1: Soft centering
    "orthogonal_he",         # Option 2: QR orthogonal (gain=sqrt(2))
    "orthogonal_tuned",      # Option 2b: QR orthogonal (gain=sqrt(1.65))
    "centered_with_dc",      # Option 3: Centered + DC
    "kernel_preserving",     # Option 4: Optimization-based
]

INIT_LABELS = {
    "uniform": "Uniform",
    "he": "He (Gaussian)", 
    "row_centered": "Row-Centered",
    "row_centered_var_adj": "Row-Centered (Var Adj)",
    "partial_centered": "Partial Centered (α=0.5)",
    "orthogonal_he": "Orthogonal He",
    "orthogonal_tuned": "Orthogonal Tuned",
    "centered_with_dc": "Centered + DC",
    "kernel_preserving": "Kernel Preserving",
}

# Reset seed for fair comparison
np.random.seed(42)

# Run experiments for each initialization
results_by_init = {init_type: {} for init_type in INIT_TYPES}

for init_type in INIT_TYPES:
    print(f"\nRunning {INIT_LABELS[init_type]} initialization...")
    np.random.seed(42)  # Reset seed for each init type for fair comparison
    
    for n_layers in LAYER_COUNTS:
        X_proj = multi_layer_square_with_init(X, n_layers, init_type=init_type)
        
        # Reduce for visualization
        pca_temp = PCA(n_components=2)
        X_vis = pca_temp.fit_transform(X_proj)
        results_by_init[init_type][n_layers] = X_vis
        
        print(f"  {n_layers} layers: done")

In [ ]:
# Plot comparison: rows = initialization types, columns = layer counts
fig, axes = plt.subplots(len(INIT_TYPES), len(LAYER_COUNTS), figsize=(4*len(LAYER_COUNTS), 4*len(INIT_TYPES)))

scatter_kwargs = dict(s=3, c=y, cmap='viridis', alpha=0.7)

for i, init_type in enumerate(INIT_TYPES):
    for j, n_layers in enumerate(LAYER_COUNTS):
        ax = axes[i, j]
        X_vis = results_by_init[init_type][n_layers]
        ax.scatter(X_vis[:, 0], X_vis[:, 1], **scatter_kwargs)
        
        if i == 0:
            ax.set_title(f"{n_layers} Layers", fontsize=12)
        if j == 0:
            ax.set_ylabel(f"{INIT_LABELS[init_type]}", fontsize=12)
        
        ax.axis('equal')
        ax.set_xticks([])
        ax.set_yticks([])

plt.suptitle(f"{DATASET.upper()}: Effect of Initialization on Multi-Layer Square RP + ReLU Geometry", 
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Alternative view: For each layer count, show all initializations side by side
for n_layers in LAYER_COUNTS:
    fig, axes = plt.subplots(1, len(INIT_TYPES), figsize=(5*len(INIT_TYPES), 5))
    
    for i, init_type in enumerate(INIT_TYPES):
        ax = axes[i]
        X_vis = results_by_init[init_type][n_layers]
        ax.scatter(X_vis[:, 0], X_vis[:, 1], **scatter_kwargs)
        ax.set_title(f"{INIT_LABELS[init_type]}", fontsize=12)
        ax.axis('equal')
    
    plt.suptitle(f"{n_layers} Layers: Comparing Initialization Strategies", fontsize=14)
    plt.tight_layout()
    plt.show()

## 5. Johnson-Lindenstrauss Comparison

Compare JL projection (distance-preserving) with RP + ReLU.

In [ ]:
# JL projection
X_jl = jl_projection(X, target_dim=2, apply_relu=False)

# RP (same target dim)
X_rp_2d = apply_random_projection(X, d_out=2, apply_relu=False)
X_rp_2d_relu = apply_random_projection(X, d_out=2, apply_relu=True)

# Plot comparison
fig = plot_jl_vs_rp(
    X_jl, X_rp_2d, X_rp_2d_relu,
    labels=y,
    title=f"{DATASET.upper()}: JL vs RP vs RP+ReLU"
)
plt.show()

## 6. GPU-Accelerated Multi-Layer (Optional)

Use PyTorch for faster computation on larger experiments.

In [ ]:
# Convert to torch tensor
X_tensor = torch.from_numpy(X).float().to(DEVICE)

# Run multi-layer projection on GPU
print(f"Running on {DEVICE}...")
for n_layers in [10, 20, 50]:
    X_proj = multi_layer_projection(
        X_tensor, n_layers, 
        mode="rectangular", 
        output_dim=2,
        device=DEVICE
    )
    print(f"  {n_layers} layers: output shape {X_proj.shape}")
    
    # Move back to CPU for plotting
    X_np = X_proj.cpu().numpy()
    
    plt.figure(figsize=(8, 6))
    plt.scatter(X_np[:, 0], X_np[:, 1], c=y, s=5, cmap='viridis', alpha=0.7)
    plt.title(f"{n_layers} layers (GPU-accelerated)")
    plt.axis('equal')
    plt.show()

## Summary

Key observations:
1. Random projections provide different structure than PCA
2. ReLU activation fundamentally changes the projection behavior
3. Rectangle vs square projections have different characteristics
4. Multiple layers progressively transform the data representation
5. JL projections aim to preserve distances, while RP+ReLU transforms the geometry